In [1]:
# Cell 36: Understand GP data sources
import pandas as pd
import numpy as np

# Check visual_location — this gives 3D body center per cow per timestamp
cow1 = pd.read_csv("/home/oussema/pfa-mmcows/store/data/raw/visual_data/visual_location/C01/C01_0725.csv")
print("=== Visual location (cow C01) ===")
print(cow1.head())
print(f"Shape: {cow1.shape}")

# Check projection matrix format
import xml.etree.ElementTree as ET
tree = ET.parse("/home/oussema/pfa-mmcows/store/data/raw/visual_data/proj_mat/0725/proj_mat_cam1.xml")
root = tree.getroot()
proj_data = root.find('.//projection_matrix/data').text.strip().split()
proj_mat = np.array([float(x) for x in proj_data]).reshape(3, 4)
print("\n=== Projection matrix cam1 (3x4) ===")
print(proj_mat)

=== Visual location (cow C01) ===
      timestamp  loc_x_cm  loc_y_cm  loc_z_cm
0  1.690272e+09    -797.0    -571.0      87.0
1  1.690272e+09    -790.0    -558.0      82.0
2  1.690272e+09    -802.0    -580.0     102.0
3  1.690272e+09    -791.0    -571.0      85.0
4  1.690272e+09    -794.0    -570.0      82.0
Shape: (5040, 4)

=== Projection matrix cam1 (3x4) ===
[[ 4.88892741e-05 -1.46994992e-03 -1.27567973e-04  9.03800210e-01]
 [-5.98769578e-05 -9.95622737e-05 -1.16241732e-03  4.27950136e-01]
 [ 3.28271746e-07 -2.39893382e-07 -1.10542608e-07  5.62836849e-04]]


In [2]:
# Cell 37: Test projection — 3D world → 2D image pixel
import numpy as np

# Projection matrix for cam1 (3x4)
proj_mat = np.array([
    [4.88892741e-05, -1.46994992e-03, -1.27567973e-04, 9.03800210e-01],
    [-5.98769578e-05, -9.95622737e-05, -1.16241732e-03, 4.27950136e-01],
    [3.28271746e-07, -2.39893382e-07, -1.10542608e-07, 5.62836849e-04]
])

# Cow C01 at first timestamp: 3D body center
body_3d = np.array([-797.0, -571.0, 87.0])

# Step 1: Set Z=0 to get World Ground Point (W-GP)
# The paper says: project 3D body center onto ground plane by setting Z=0
wgp = np.array([-797.0, -571.0, 0.0, 1.0])  # [X, Y, Z=0, 1] homogeneous

# Step 2: Project W-GP into camera image using projection matrix
# result = P × [X, Y, Z, 1]^T → gives [u*w, v*w, w]
# Then pixel = (u*w/w, v*w/w) = (u, v)
pixel_h = proj_mat @ wgp  # homogeneous 2D point
pixel_x = pixel_h[0] / pixel_h[2]  # divide by w to get actual pixel x
pixel_y = pixel_h[1] / pixel_h[2]  # divide by w to get actual pixel y

print(f"3D body center: {body_3d}")
print(f"World Ground Point (Z=0): {wgp[:3]}")
print(f"Projected 2D GP in cam1: ({pixel_x:.1f}, {pixel_y:.1f})")
print(f"\nImage size: 4480 x 2800")
print(f"GP in range? x: {0 <= pixel_x <= 4480}, y: {0 <= pixel_y <= 2800}")

3D body center: [-797. -571.   87.]
World Ground Point (Z=0): [-797. -571.    0.]
Projected 2D GP in cam1: (3889.2, 1215.3)

Image size: 4480 x 2800
GP in range? x: True, y: True


In [3]:
# Cell 38: Generate YOLOv8Pose annotations with Ground Points
#
# For each image, we:
# 1. Read existing bbox label (cow_id, xc, yc, w, h)
# 2. Find that cow's 3D location at that timestamp from visual_location
# 3. Set Z=0 → World Ground Point
# 4. Project W-GP into camera → 2D GP pixel
# 5. Normalize GP to [0,1] range
# 6. Write new label: 0 xc yc w h gp_x_norm gp_y_norm 2
#    (class=0, then bbox, then keypoint, 2=visible)

import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET
from pathlib import Path

DATA_ROOT = Path("/home/oussema/pfa-mmcows/store/data/raw/visual_data")
LABELS_ORIG = DATA_ROOT / "labels" / "combined" / "0725"
IMAGES = DATA_ROOT / "images" / "0725"
VIS_LOC = DATA_ROOT / "visual_location"
PROJ_DIR = DATA_ROOT / "proj_mat" / "0725"

CAMERAS = ["cam_1", "cam_2", "cam_3", "cam_4"]
IMG_W, IMG_H = 4480, 2800

# Output: new pose labels
POSE_LABELS = Path("/home/oussema/pfa-mmcows/store/data/clean/yolo_pose_labels/0725")

# --- Load projection matrices ---
# proj_mat_cam1.xml → cam_1, etc.
proj_mats = {}
for cam in CAMERAS:
    cam_num = cam.split("_")[1]  # "1", "2", etc.
    xml_path = PROJ_DIR / f"proj_mat_cam{cam_num}.xml"
    tree = ET.parse(xml_path)
    data = tree.find('.//projection_matrix/data').text.strip().split()
    proj_mats[cam] = np.array([float(x) for x in data]).reshape(3, 4)

# --- Load visual_location for all cows ---
# Cow IDs in labels are 0-15, cow folders are C01-C16
cow_locs = {}
for cow_id in range(16):
    cow_name = f"C{cow_id+1:02d}"  # 0 → C01, 15 → C16
    csv_path = VIS_LOC / cow_name / f"{cow_name}_0725.csv"
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        # Index by rounded timestamp for fast lookup
        df["ts_round"] = df["timestamp"].round(0).astype(int)
        cow_locs[cow_id] = df.set_index("ts_round")

print(f"Loaded {len(cow_locs)} cow location files")
print(f"Loaded {len(proj_mats)} projection matrices")

Loaded 16 cow location files
Loaded 4 projection matrices


In [4]:
def project_to_gp(loc_x, loc_y, proj_mat):
    """Project world ground point (Z=0) to 2D image pixel."""
    wgp = np.array([loc_x, loc_y, 0.0, 1.0])
    pixel_h = proj_mat @ wgp
    if abs(pixel_h[2]) < 1e-10:
        return None, None
    px = pixel_h[0] / pixel_h[2]
    py = pixel_h[1] / pixel_h[2]
    return px, py

In [5]:
# --- Generate pose labels ---
stats = {"total": 0, "with_gp": 0, "no_gp": 0}

for cam in CAMERAS:
    out_dir = POSE_LABELS / cam
    out_dir.mkdir(parents=True, exist_ok=True)
    
    images = sorted((IMAGES / cam).glob("*.jpg"))
    
    for img_path in images:
        # Extract timestamp from filename: 1690271846_02-57-26.jpg → 1690271846
        ts = int(img_path.stem.split("_")[0])
        
        # Read original label
        lbl_path = LABELS_ORIG / cam / (img_path.stem + ".txt")
        if not lbl_path.exists():
            continue
        
        new_lines = []
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cow_id = int(parts[0])
                xc, yc, w, h = parts[1], parts[2], parts[3], parts[4]
                
                stats["total"] += 1
                
                # Find cow's 3D location at this timestamp
                if cow_id in cow_locs and ts in cow_locs[cow_id].index:
                    row = cow_locs[cow_id].loc[ts]
                    if isinstance(row, pd.DataFrame):
                        row = row.iloc[0]
                    
                    px, py = project_to_gp(row["loc_x_cm"], row["loc_y_cm"], proj_mats[cam])
                    
                    if px is not None and 0 <= px <= IMG_W and 0 <= py <= IMG_H:
                        # Normalize GP to [0,1]
                        gp_x_norm = px / IMG_W
                        gp_y_norm = py / IMG_H
                        # Format: class xc yc w h gp_x gp_y visibility
                        # visibility: 2 = visible
                        new_lines.append(f"0 {xc} {yc} {w} {h} {gp_x_norm:.6f} {gp_y_norm:.6f} 2")
                        stats["with_gp"] += 1
                    else:
                        # GP out of image — keep bbox only with invisible keypoint
                        new_lines.append(f"0 {xc} {yc} {w} {h} 0.0 0.0 0")
                        stats["no_gp"] += 1
                else:
                    # No location data — keep bbox with invisible keypoint
                    new_lines.append(f"0 {xc} {yc} {w} {h} 0.0 0.0 0")
                    stats["no_gp"] += 1
        
        # Write new label
        out_path = out_dir / (img_path.stem + ".txt")
        with open(out_path, "w") as f:
            f.write("\n".join(new_lines) + "\n")

print(f"\n✅ Pose labels generated:")
print(f"  Total annotations: {stats['total']}")
print(f"  With GP: {stats['with_gp']} ({100*stats['with_gp']/stats['total']:.1f}%)")
print(f"  Without GP: {stats['no_gp']} ({100*stats['no_gp']/stats['total']:.1f}%)")
print(f"  Output: {POSE_LABELS}")

# Verify a sample
print("\n=== Sample pose label ===")
sample = list((POSE_LABELS / "cam_1").glob("*.txt"))[0]
print(f"File: {sample.name}")
with open(sample) as f:
    print(f.read()[:500])


✅ Pose labels generated:
  Total annotations: 213705
  With GP: 196974 (92.2%)
  Without GP: 16731 (7.8%)
  Output: /home/oussema/pfa-mmcows/store/data/clean/yolo_pose_labels/0725

=== Sample pose label ===
File: 1690334231_20-17-11.txt
0 0.7455357142857143 0.27517857142857144 0.09464285714285714 0.10892857142857143 0.297392 0.271740 2
0 0.5294642857142857 0.1875 0.06160714285714286 0.1 0.389081 0.232528 2
0 0.3366071428571429 0.31089285714285714 0.15267857142857144 0.19535714285714287 0.275725 0.248392 2
0 0.7717633928571429 0.3539285714285714 0.12745535714285713 0.17214285714285715 0.327052 0.299650 2
0 0.39296875 0.16517857142857142 0.09709821428571429 0.08821428571428572 0.247147 0.210348 2
0 0.32120535714285714 0.258392857


In [8]:
# Cell 39: Build YOLOv8Pose dataset structure
#
# Same logic as A2 but with pose labels:
# - Symlink images (no copy)
# - Copy pose labels (with GP keypoints)
# - Use same temporal split from dataset_index.csv
# - Generate pose dataset YAML

import pandas as pd
from pathlib import Path

INDEX = pd.read_csv("/home/oussema/mmcows-visual/notebooks/yolo_dataset_index.csv")
IMAGES_SRC = Path("/home/oussema/pfa-mmcows/store/data/raw/visual_data/images/0725")
POSE_LABELS_SRC = Path("/home/oussema/pfa-mmcows/store/data/clean/yolo_pose_labels/0725")
POSE_ROOT = Path("/home/oussema/pfa-mmcows/store/data/clean/yolo_pose_dataset")

# Create folder structure
for split in ["train", "val", "test"]:
    (POSE_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (POSE_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

count = {"train": 0, "val": 0, "test": 0}

for _, row in INDEX.iterrows():
    split = row["split"]
    cam = row["camera"]
    fname = Path(row["image_path"]).stem
    
    # Symlink image
    img_src = Path(row["image_path"])
    img_dst = POSE_ROOT / "images" / split / f"{cam}_{fname}.jpg"
    if not img_dst.exists():
        img_dst.symlink_to(img_src)
    
    # Copy pose label
    lbl_src = POSE_LABELS_SRC / cam / (fname + ".txt")
    lbl_dst = POSE_ROOT / "labels" / split / f"{cam}_{fname}.txt"
    if lbl_src.exists() and not lbl_dst.exists():
        import shutil
        shutil.copy2(lbl_src, lbl_dst)
    
    count[split] += 1

# Generate YAML
# kpt_shape: [1, 3] means 1 keypoint with 3 values (x, y, visibility)
yaml_content = f"""path: {POSE_ROOT}
train: images/train
val: images/val
test: images/test

nc: 1
names: ['cow']

kpt_shape: [1, 3]
"""

(POSE_ROOT / "dataset_pose.yaml").write_text(yaml_content)

print("✅ Pose dataset built:")
for s, n in count.items():
    print(f"  {s}: {n} images")
print(f"\nYAML: {POSE_ROOT / 'dataset_pose.yaml'}")

✅ Pose dataset built:
  train: 14106 images
  val: 3024 images
  test: 3023 images

YAML: /home/oussema/pfa-mmcows/store/data/clean/yolo_pose_dataset/dataset_pose.yaml


## Training

In [11]:
# Cell 40: Train YOLOv8Pose-nano
#
# Same as A3 but with YOLOv8Pose-nano:
# - Detects bounding boxes AND predicts Ground Point keypoint
# - Think of it like: the model now learns TWO things at once:
#   "where is the cow?" (box) AND "where are its feet on the ground?" (GP)
# - The Pose model adds a keypoint head on top of the detection head
# - Loss = box_loss + cls_loss + dfl_loss + pose_loss + kobj_loss
#   (two new losses for keypoint accuracy and keypoint objectness)

from ultralytics import YOLO

model = YOLO("yolov8n-pose.pt")  # Pose variant of nano

model.train(
    data="/home/oussema/pfa-mmcows/store/data/clean/yolo_pose_dataset/dataset_pose.yaml",
    epochs=50,
    imgsz=640,
    batch=16,          # max batch for GPU
    device=0,
    workers=8,
    amp=True,          # mixed precision
    project="/home/oussema/pfa-mmcows/runs/pose",
    name="yolov8n_pose_cows",
    patience=10,
    save=True,
    plots=True,
)

New https://pypi.org/project/ultralytics/8.4.46 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.39 🚀 Python-3.13.12 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 7706MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/oussema/pfa-mmcows/store/data/clean/yolo_pose_dataset/dataset_pose.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=y

: 